In [23]:
import cv2
import math
import mediapipe as mp
import numpy as np

# ---------------- Mediapipe Setup ----------------
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode
ImageFormat = mp.ImageFormat
Image = mp.Image

# ---------------- Thresholds ----------------
NECK_GOOD = 20
TORSO_GOOD = 10
SMOOTHING = 20

LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_EAR = 7
LEFT_HIP = 23

# ---------------- Utility ----------------
def calculate_angle(x1, y1, x2, y2):
    dx = x2 - x1
    dy = y1 - y2
    return abs(math.degrees(math.atan2(dx, dy)))

# ---------------- Analyzer ----------------
class PostureAnalyzer:
    def __init__(self):
        model = r"D:\assortments\posture\pose_landmarker_lite.task"
        options = PoseLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=model),
            running_mode=VisionRunningMode.VIDEO
        )
        self.landmarker = PoseLandmarker.create_from_options(options)

        self.colors = {
            "good": (80,180,90),
            "warn": (0,220,255),
            "bad": (50,50,230),
            "panel": (245,245,245),
            "text": (40,40,40),
            "muted": (200,200,200),
            "skeleton": (90,90,90)
        }

        self.good_frames = 0
        self.bad_frames = 0
        self.neck_hist = []
        self.torso_hist = []
        self.display_score = 100

        # fixed top/bottom
        self.top_height = 80
        self.bottom_height = 160

    # ---------------- Session Score ----------------
    def session_score(self):
        total = self.good_frames + self.bad_frames
        return int((self.good_frames / total) * 100) if total else 100

    # ---------------- Process Frame ----------------
    def process(self, frame, timestamp):
        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = Image(image_format=ImageFormat.SRGB, data=rgb)
        result = self.landmarker.detect_for_video(mp_image, timestamp)

        if not result.pose_landmarks:
            return self.compose_ui(frame, "NO DATA", 0, 0, 0, self.colors["bad"])

        lm = result.pose_landmarks[0]

        # Key points
        l_sh = (int(lm[LEFT_SHOULDER].x*w), int(lm[LEFT_SHOULDER].y*h))
        r_sh = (int(lm[RIGHT_SHOULDER].x*w), int(lm[RIGHT_SHOULDER].y*h))
        ear = (int(lm[LEFT_EAR].x*w), int(lm[LEFT_EAR].y*h))
        hip = (int(lm[LEFT_HIP].x*w), int(lm[LEFT_HIP].y*h))
        sh = ((l_sh[0]+r_sh[0])//2, (l_sh[1]+r_sh[1])//2)

        # Angles
        neck = calculate_angle(*sh,*ear)
        torso = calculate_angle(*hip,*sh)

        # Smooth angles
        self.neck_hist.append(neck)
        self.torso_hist.append(torso)
        if len(self.neck_hist) > SMOOTHING:
            self.neck_hist.pop(0)
            self.torso_hist.pop(0)
        neck = sum(self.neck_hist)/len(self.neck_hist)
        torso = sum(self.torso_hist)/len(self.torso_hist)

        # Score
        neck_pen = max(0, neck - NECK_GOOD)
        torso_pen = max(0, torso - TORSO_GOOD)
        penalty = neck_pen*0.4 + torso_pen*0.6
        score = max(0,100-int(penalty*2))
        self.display_score = self.display_score*0.9 + score*0.1
        score = int(self.display_score)

        if score > 80:
            status = "ALIGNED"
            color = self.colors["good"]
            self.good_frames += 1
        elif score > 60:
            status = "WARNING"
            color = self.colors["warn"]
            self.bad_frames += 1
        else:
            status = "MISALIGNED"
            color = self.colors["bad"]
            self.bad_frames += 1

        self.draw_skeleton(frame, lm, w, h)
        frame = self.compose_ui(frame, status, score, neck, torso, color)
        return frame

    # ---------------- Draw Skeleton ----------------
    def draw_skeleton(self, frame, lm, w, h):
        c = self.colors["skeleton"]
        connections = [
            (11,12),(11,13),(13,15),(12,14),(14,16),(11,23),(12,24),(23,24)
        ]
        for a,b in connections:
            x1 = int(lm[a].x*w); y1 = int(lm[a].y*h)
            x2 = int(lm[b].x*w); y2 = int(lm[b].y*h)
            cv2.line(frame,(x1,y1),(x2,y2),c,2)
        for i in [7,11,12,23]:
            x = int(lm[i].x*w); y = int(lm[i].y*h)
            cv2.circle(frame,(x,y),5,c,-1)

    # ---------------- Compose UI ----------------
    def compose_ui(self, frame, status, score, neck, torso, color):
        h, w = frame.shape[:2]
        top = self.top_height
        bottom = self.bottom_height
        canvas = np.zeros((h+top+bottom, w, 3), dtype=np.uint8)
        canvas[:] = self.colors["panel"]
        canvas[top:top+h] = frame
        self.draw_top_panel(canvas, status, score, color, w, top)
        self.draw_bottom_panel(canvas, neck, torso, w, h+top)
        return canvas

    # ---------------- Top Panel ----------------
    def draw_top_panel(self, canvas, status, score, color, w, top_height):
        padding = 40
        text = f"{status}"
        (text_width, text_height), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        text_x = padding
        text_y = top_height // 2 + text_height // 2
        cv2.putText(canvas, text, (text_x, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

        # Place the ring at right with margin
        ring_margin = 40
        ring_x = w - 40
        ring_y = top_height // 2
        self.draw_score_ring(canvas, score, ring_x, ring_y)

    # ---------------- Score Ring ----------------
    def draw_score_ring(self, frame, score, cx, cy):
        r=30
        cv2.circle(frame,(cx,cy),r,self.colors["muted"],6)
        angle=int(360*score/100)
        cv2.ellipse(frame,(cx,cy),(r,r),-90,0,angle,(80,180,90),6)
        cv2.putText(frame,str(score),(cx-18,cy+8),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,(30,30,30),2)

    # ---------------- Bottom Panel ----------------
    def draw_bottom_panel(self, canvas, neck, torso, w, y):
        session = self.session_score()
        cv2.putText(canvas, f"SESSION SCORE: {session}%", (60, y+40),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,self.colors["text"],2)
        self.draw_bar(canvas, "Neck Angle", neck, NECK_GOOD, 60, y+80)
        self.draw_bar(canvas, "Torso Angle", torso, TORSO_GOOD, 60, y+120)

    # ---------------- Responsive Bar ----------------
    def draw_bar(self, frame, label, value, limit, x, y):
        max_w = int(frame.shape[1]*0.45)
        h = 20
        ratio = min(value/(limit*2),1)
        if value <= limit:
            color = self.colors["good"]
        elif value <= limit*1.5:
            color = self.colors["warn"]
        else:
            color = self.colors["bad"]
        w_rect = int(max_w*ratio)
        cv2.rectangle(frame,(x,y),(x+max_w,y+h),(220,220,220),-1)
        cv2.rectangle(frame,(x,y),(x+w_rect,y+h),color,-1)
        cv2.putText(frame,label,(x,y-6),cv2.FONT_HERSHEY_SIMPLEX,0.55,self.colors["text"],1)
        cv2.putText(frame,f"{value:.1f}",(x+max_w+10,y+15),cv2.FONT_HERSHEY_SIMPLEX,0.5,self.colors["text"],1)

# ---------------- Main ----------------
def main():
    cap = cv2.VideoCapture("video.mov")
    if not cap.isOpened():
        print("Failed to open video")
        return

    analyzer = PostureAnalyzer()
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Use same top/bottom as analyzer
    top = analyzer.top_height
    bottom = analyzer.bottom_height

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter("newoutput.mp4", fourcc, fps, (w, h + top + bottom))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        timestamp = int(cap.get(cv2.CAP_PROP_POS_MSEC))
        frame = analyzer.process(frame, timestamp)
        out.write(frame)
        cv2.imshow("Posture Analysis", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()